# 09 Survival Analysis — Reference Solutions

Complete solutions for the survival analysis exercises on the Pine and Cypress Nursing Home Legionella outbreak.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# -- CJK font setup (avoids Chinese labels showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1].copy()
cases["event"] = (cases["outcome"] == "dead").astype(int)

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

## Question 1: Survival Analysis for CHF (Congestive Heart Failure)

In [ ]:
# KM curves: CHF vs No CHF
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("CHF", cases["comorbidity_chf"] == 1),
                     ("No CHF", cases["comorbidity_chf"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Survival curves: CHF vs No CHF")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank test
chf_yes = cases[cases["comorbidity_chf"] == 1]
chf_no = cases[cases["comorbidity_chf"] == 0]

result = logrank_test(
    chf_yes["time_to_event"], chf_no["time_to_event"],
    event_observed_A=chf_yes["event"],
    event_observed_B=chf_no["event"],
)

print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("\n→ p < 0.05: CHF significantly affects survival")
else:
    print("\n→ p ≥ 0.05: CHF's effect on survival is not statistically significant")
    print("→ Possibly because the sample is too small (only 19 deaths), so the test is underpowered")

## Question 2: Survival Comparison by Age Group

In [ ]:
# Age grouping
cases["age_group"] = np.where(cases["age"] >= 75, "≥75", "<75")

fig, ax = plt.subplots(figsize=(8, 5))

for label in ["≥75", "<75"]:
    sub = cases[cases["age_group"] == label]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"Age {label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Survival curves: Age ≥75 vs <75")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
old = cases[cases["age"] >= 75]
young = cases[cases["age"] < 75]

result_age = logrank_test(
    old["time_to_event"], young["time_to_event"],
    event_observed_A=old["event"],
    event_observed_B=young["event"],
)

print(f"Log-rank test statistic = {result_age.test_statistic:.3f}")
print(f"p-value = {result_age.p_value:.4f}")

if result_age.p_value < 0.05:
    print("\n→ The elderly (≥75) group has significantly worse survival")
else:
    print("\n→ The effect of age grouping on survival is not statistically significant")
    print("→ Nursing home residents are generally older, so the between-group difference may not be large enough")

## Question 3 (Challenge): Hospitalized vs Not Hospitalized + Cox Regression

In [ ]:
# KM curves: hospitalized vs not hospitalized
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("Hospitalized", cases["hospitalized"] == 1),
                     ("Not hospitalized", cases["hospitalized"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Survival curves: Hospitalized vs Not hospitalized")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
hosp_yes = cases[cases["hospitalized"] == 1]
hosp_no = cases[cases["hospitalized"] == 0]

result_hosp = logrank_test(
    hosp_yes["time_to_event"], hosp_no["time_to_event"],
    event_observed_A=hosp_yes["event"],
    event_observed_B=hosp_no["event"],
)

print(f"Log-rank test statistic = {result_hosp.test_statistic:.3f}")
print(f"p-value = {result_hosp.p_value:.4f}")

In [ ]:
# Cox regression
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "hospitalized", "comorbidity_copd", "comorbidity_chf",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox regression results ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

# HR forest plot
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression — HR forest plot")
plt.tight_layout()
plt.show()

print("\n=== Interpretation ===")
hosp_hr = cph.summary.loc["hospitalized", "exp(coef)"]
print(f"hospitalized HR = {hosp_hr:.3f}")
if hosp_hr > 1:
    print("→ Hospitalized patients have HR > 1, seemingly 'increasing' the risk of death")
    print("→ But this does not mean hospitalization is a risk factor!")
    print("→ People are hospitalized because they are severely ill—this is confounding by indication")
    print("→ Hospitalization is a 'marker' of severity, not a 'cause' of death")
else:
    print("→ Hospitalized patients have HR < 1; after adjusting for other factors, hospitalization may have a protective effect")
    print("→ But interpret with caution—confounding by indication still applies")

### Key Interpretation Points

- **CHF**: heart failure may increase the risk of death, but in a small sample it may not reach statistical significance
- **Age**: nursing home residents are generally older, so the between-group difference may be small
- **Hospitalization**: this is a classic case of **confounding by indication** in survival analysis
  - The mortality of hospitalized patients may be higher, but the reason is that "more severely ill people are the ones who get hospitalized"
  - Hospitalization itself is a treatment action that should reduce the risk of death
  - But in observational data, the HR for hospitalization may be > 1 because it is a marker of severity
- **Limitation**: this case has only 19 deaths; putting too many variables in a Cox model easily overfits. The recommendation is at least 10 events per variable, so keeping at most 1-2 variables is more stable

## Question 4 Solution

In [ ]:
# Data: TB treatment cohort -- compare time to cure between drug-resistant (MDR-TB) and non-resistant patients
rng = np.random.default_rng(409)
n = 400

drug_resistant = rng.binomial(1, 0.2, size=n)  # 20% have multidrug-resistant TB (MDR-TB)
age = np.clip(rng.normal(48, 16, size=n), 15, 90).round().astype(int)

# Non-resistant patients are cured in about 150 days on average; resistant patients need longer treatment, averaging about 320 days to cure
scale_cure = np.where(drug_resistant == 1, 320, 150)
duration_to_cure = rng.exponential(scale_cure)

follow_up_end = 540  # 18-month follow-up; patients still not cured beyond this are right-censored
time_to_event = np.minimum(duration_to_cure, follow_up_end)
event = (duration_to_cure <= follow_up_end).astype(int)  # 1 = cured, 0 = still not cured at end of follow-up (censored)

tb = pd.DataFrame({
    "patient_id": [f"TB{i:04d}" for i in range(n)],
    "age": age,
    "drug_resistant": drug_resistant,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM curves: MDR-TB vs non-resistant
fig, ax = plt.subplots(figsize=(8, 5))

kmf_results = {}
for label, mask in [("MDR-TB (resistant)", tb["drug_resistant"] == 1),
                     ("Non-resistant", tb["drug_resistant"] == 0)]:
    sub = tb[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, cured={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)
    kmf_results[label] = kmf

ax.set_title("TB treatment survival curves: MDR-TB vs Non-resistant (event = cure)")
ax.set_xlabel("Days since treatment start")
ax.set_ylabel("Probability not yet cured")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Log-rank test
resistant = tb[tb["drug_resistant"] == 1]
susceptible = tb[tb["drug_resistant"] == 0]

result = logrank_test(
    resistant["time_to_event"], susceptible["time_to_event"],
    event_observed_A=resistant["event"],
    event_observed_B=susceptible["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

# Median time to cure
for label, kmf in kmf_results.items():
    print(f"{label} median time to cure: {kmf.median_survival_time_:.1f} days")

print("\n=== Interpretation ===")
if result.p_value < 0.05:
    print("→ p < 0.05: drug-resistant (MDR-TB) patients take significantly longer to be cured")
    print("→ MDR-TB requires longer treatment courses and more intensive care, and also raises the risk of treatment interruption")
else:
    print("→ The difference in time to cure between the two groups is not statistically significant")

## Question 5 Solution

In [ ]:
# Data: COVID-19 hospitalization cohort -- compare time from admission to death between ICU and general ward patients
rng = np.random.default_rng(519)
n = 500

age = np.clip(rng.normal(60, 18, size=n), 18, 95).round().astype(int)
is_male = rng.binomial(1, 0.5, size=n)
icu_admission = rng.binomial(1, 0.22, size=n)
diabetes = rng.binomial(1, 0.25, size=n)

baseline_hazard = 1 / 420  # Baseline risk of death for a 60-year-old, non-diabetic, general ward patient
linear_pred = 1.0 * icu_admission + 0.4 * diabetes + 0.03 * (age - 60)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 60  # 60-day follow-up; patients still hospitalized or discharged beyond this are right-censored
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = died during hospitalization, 0 = discharged or end of follow-up (censored)

covid = pd.DataFrame({
    "patient_id": [f"CV{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "icu_admission": icu_admission,
    "diabetes": diabetes,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM curves: ICU vs general ward
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("ICU", covid["icu_admission"] == 1),
                     ("General ward", covid["icu_admission"] == 0)]:
    sub = covid[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("COVID-19 hospitalization survival curves: ICU vs General ward")
ax.set_xlabel("Days since admission")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank test
icu_yes = covid[covid["icu_admission"] == 1]
icu_no = covid[covid["icu_admission"] == 0]

result = logrank_test(
    icu_yes["time_to_event"], icu_no["time_to_event"],
    event_observed_A=icu_yes["event"],
    event_observed_B=icu_no["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

# Cox regression
cph = CoxPHFitter()
cph.fit(covid[["time_to_event", "event", "age", "is_male", "icu_admission", "diabetes"]],
        duration_col="time_to_event", event_col="event")

print("\n=== Cox regression results ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n=== Interpretation ===")
icu_hr = cph.summary.loc["icu_admission", "exp(coef)"]
print(f"icu_admission HR = {icu_hr:.3f}")
print("→ The HR for ICU admission is significantly greater than 1, but this does not mean ICU care itself is 'harmful'")
print("→ The ICU is a resource reserved for the most severely ill patients -- this is a classic case of confounding by indication")
print("→ When interpreting HRs from observational data, always consider the question 'why was this patient admitted to the ICU?'")

## Question 6 Solution

In [ ]:
# Data: measles case contacts -- compare time from exposure to onset between vaccinated and unvaccinated contacts
rng = np.random.default_rng(626)
n = 350

vaccinated = rng.binomial(1, 0.6, size=n)  # 60% of contacts were previously vaccinated against measles
age = np.clip(rng.normal(10, 8, size=n), 0, 60).round().astype(int)

# Unvaccinated contacts have a high probability of infection (high attack rate); most vaccinated contacts are protected, so breakthrough infection is rare
p_infected = np.where(vaccinated == 1, 0.12, 0.85)
infected = rng.binomial(1, p_infected)

# If infected, the incubation period (exposure to onset) is about 10-14 days; uninfected contacts do not develop symptoms during the observation period
incubation = np.clip(rng.normal(12, 2.2, size=n), 5, 21)
surveillance_end = 21  # Contacts are followed for 21 days

time_to_event = np.where(infected == 1, incubation, surveillance_end)
event = infected  # 1 = developed symptoms within observation period, 0 = no symptoms by end of observation (censored)

measles = pd.DataFrame({
    "contact_id": [f"MS{i:04d}" for i in range(n)],
    "age": age,
    "vaccinated": vaccinated,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM curves: unvaccinated vs vaccinated
fig, ax = plt.subplots(figsize=(8, 5))

km_results = {}
for label, mask in [("Unvaccinated", measles["vaccinated"] == 0),
                     ("Vaccinated", measles["vaccinated"] == 1)]:
    sub = measles[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, onset={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)
    km_results[label] = kmf

ax.set_title("Measles contact survival curves: Unvaccinated vs Vaccinated (event = onset)")
ax.set_xlabel("Days since exposure")
ax.set_ylabel("Probability without onset")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank test
unvax = measles[measles["vaccinated"] == 0]
vax = measles[measles["vaccinated"] == 1]

result = logrank_test(
    unvax["time_to_event"], vax["time_to_event"],
    event_observed_A=unvax["event"],
    event_observed_B=vax["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

print(f"\nMedian time to onset in the unvaccinated group: {km_results['Unvaccinated'].median_survival_time_:.1f} days (approximates the median incubation period of measles)")

print("\n=== Interpretation ===")
if result.p_value < 0.05:
    print("→ p < 0.05: vaccination significantly reduces or delays onset among contacts")
    print("→ The median time to onset in the unvaccinated group can serve as a reference for the measles incubation period, informing contact tracing and quarantine duration")
else:
    print("→ The difference in time to onset between the two groups is not statistically significant")

## Question 7 Solution

In [ ]:
# Data: dengue case cohort -- compare time to progression to severe disease between secondary and primary infection patients
rng = np.random.default_rng(727)
n = 450

secondary_infection = rng.binomial(1, 0.35, size=n)  # 35% had a secondary infection (previously infected with a different dengue serotype)
age = np.clip(rng.normal(32, 16, size=n), 1, 85).round().astype(int)
is_male = rng.binomial(1, 0.48, size=n)

baseline_hazard = 1 / 45  # Baseline risk of progression to severe disease for a 32-year-old primary infection patient
linear_pred = 1.1 * secondary_infection + 0.012 * (age - 32)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 14  # 14-day clinical follow-up period after onset
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = progressed to severe dengue, 0 = did not progress by end of follow-up (censored)

dengue = pd.DataFrame({
    "case_id": [f"DF{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "secondary_infection": secondary_infection,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# KM curves: secondary vs primary infection
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("Secondary infection", dengue["secondary_infection"] == 1),
                     ("Primary infection", dengue["secondary_infection"] == 0)]:
    sub = dengue[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, severe={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Dengue progression survival curves: Secondary infection vs Primary infection")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Probability without progression to severe disease")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank test
secondary = dengue[dengue["secondary_infection"] == 1]
primary = dengue[dengue["secondary_infection"] == 0]

result = logrank_test(
    secondary["time_to_event"], primary["time_to_event"],
    event_observed_A=secondary["event"],
    event_observed_B=primary["event"],
)
print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

# Cox regression
cph = CoxPHFitter()
cph.fit(dengue[["time_to_event", "event", "age", "is_male", "secondary_infection"]],
        duration_col="time_to_event", event_col="event")

print("\n=== Cox regression results ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

print("\n=== Interpretation ===")
hr = cph.summary.loc["secondary_infection", "exp(coef)"]
p_val = cph.summary.loc["secondary_infection", "p"]
print(f"secondary_infection HR = {hr:.3f}, p = {p_val:.4f}")
if p_val < 0.05 and hr > 1:
    print("→ Secondary infection significantly increases the hazard ratio for progression to severe disease")
    print("→ This is consistent with the antibody-dependent enhancement (ADE) hypothesis:")
    print("  non-neutralizing pre-existing antibodies may help the virus enter cells, worsening the disease course")
else:
    print("→ The effect of secondary infection on the risk of progression to severe disease is not statistically significant")

## Question 8 Solution

In [ ]:
# Data: continue using cases loaded at the start of this chapter (Pine and Cypress Nursing Home Legionella cases, event = death)
cox_cols_q8 = ["time_to_event", "event", "age", "sex",
               "icu_admission", "immunosuppressed", "comorbidity_cancer"]
cox_df8 = cases[cox_cols_q8].copy()
cox_df8["is_male"] = (cox_df8["sex"] == "M").astype(int)
cox_df8 = cox_df8.drop(columns=["sex"])

print(cox_df8.describe().round(2))

# Cox regression
cph8 = CoxPHFitter()
cph8.fit(cox_df8, duration_col="time_to_event", event_col="event")

print("\n=== Cox regression results ===")
summary8 = cph8.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary8.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary8.round(3).to_string())

print("\n=== Proportional hazards assumption check ===")
try:
    cph8.check_assumptions(cox_df8, p_value_threshold=0.05, show_plots=False)
except Exception as exc:
    print(f"(The proportional hazards test may be unstable with a small sample: {exc})")

c_index = cph8.concordance_index_
print(f"\nC-index (concordance index) = {c_index:.3f}")

print("\n=== Interpretation ===")
sig_vars = summary8[summary8["p_value"] < 0.05].index.tolist()
if sig_vars:
    print(f"→ Factors that significantly increase the risk of death: {', '.join(sig_vars)}")
else:
    print("→ No variable in this model reached statistical significance (p < 0.05)")
print("→ With only 19 deaths and 5 covariates in the model, events per variable is low,")
print("  so confidence intervals may be wide and the model prone to overfitting -- interpret the results with caution")
if c_index > 0.7:
    print(f"→ C-index = {c_index:.3f} > 0.7, the model has reasonably good discrimination")
else:
    print(f"→ C-index = {c_index:.3f}, the model's discrimination is limited and more data is needed for validation")